"MLflow solves this in three ways."


"First — Experiment Tracking. Every model run gets logged automatically. Parameters, metrics, artifacts. Timestamped. Named. Searchable. You never lose a result again."


"Second — Model Registry. Your best model gets registered with a version number. Version 1 goes to Staging. After validation it gets promoted to Production. Your serving layer always knows which version is live."


"Third — Reproducibility. Any run can be re-executed exactly. Same parameters, same data version, same result. That's the production guarantee."

In [1]:
# WITHOUT MLflow                  WITH MLflow
# ─────────────────               ─────────────────
# print("accuracy: 0.95")         mlflow.log_metric("accuracy", 0.95)
# where did I save this?        → stored, searchable, versioned
# what params did I use?        → params logged automatically
# which was the best run?       → UI comparison in one click
# can I reproduce run 3?        → yes, run ID captures everything

# Phase 4 — MLflow Experiment Tracking
## Healthcare AI System

**Prerequisites:**
- MLflow UI running at http://127.0.0.1:5000
- model_table.csv in outputs/ folder

**Run in terminal before this notebook:**
```bash
cd Healthcare
mlflow ui
```

In [2]:
import mlflow
import os
import joblib
import pandas as pd
import warnings
import json
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, f1_score, recall_score)
# Point MLFLOW to project root 
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print("MLflow Version: ", mlflow.__version__)
print("Tracking URI: ", mlflow.get_tracking_uri())

MLflow Version:  3.16.0
Tracking URI:  sqlite:///../mlflow.db


In [3]:
# Set the Experiment
mlflow.set_experiment("healthcare-risk-classification")
print("Experiment Created ✓")

Experiment Created ✓


In [4]:
# Step 1) Load the saved Models
risk_rf_model = joblib.load("../models/risk_model.joblib")
claim_rf_model = joblib.load("../models/claim_model.joblib")
print("Models Loaded ✓")
print("risk_model  :", type(risk_rf_model))
print("claim_model :", type(claim_rf_model))

Models Loaded ✓
risk_model  : <class 'sklearn.pipeline.Pipeline'>
claim_model : <class 'sklearn.pipeline.Pipeline'>


In [5]:
# Step 2) Load the dataset
df = pd.read_csv("../outputs/model_table.csv", parse_dates=["registration_date", "visit_date", "billing_date"])
print("Data Loaded ✓")
print("Shape: ", df.shape)
df.head()

Data Loaded ✓
Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,3993,9,M,Hyderabad,HealthPlus,1,2025-10-18,684,2025-01-20,General,...,0,0,0,271,4,12.645000,0.149678,1,0,1
1,76,59,M,Delhi,HealthPlus,0,2025-07-09,1412,2025-01-20,General,...,1,1,0,170,5,22.330000,0.149678,1,0,0
2,3393,43,M,Hyderabad,HealthPlus,0,2025-07-10,1510,2025-01-20,ICU,...,1,2,1,171,8,24.091250,0.149678,1,0,0
3,1998,29,M,Bangalore,MediCareX,1,2025-04-22,1549,2025-01-20,Cardiology,...,1,0,0,92,11,18.974545,0.152480,1,0,1
4,3500,38,M,Hyderabad,CareOne,1,2025-08-02,2275,2025-01-20,Orthopedics,...,0,1,0,194,7,21.358571,0.148655,1,0,0


In [6]:
# Step 3) Load Feature Schema 
with open("../outputs/feature_schema.json", "r") as f:
    schema = json.load(f)

risk_features = schema["risk_model_features"]
claim_features = schema["claim_model_features"]
risk_target = schema["risk_target"]
claim_target = schema["claim_target"]

print("Schema loaded ✓")
print(f"Risk  features : {len(risk_features)}")
print(f"Claim features : {len(claim_features)}")

Schema loaded ✓
Risk  features : 14
Claim features : 18


In [7]:
# Step 4) Create separate datasets
risk_df = df.copy()
claim_df = df.copy()

In [8]:
# Step 5) Time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)

split_idx = int(len(risk_df) * 0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test  = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk  = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk  = risk_test[risk_target]

print("Risk Train shape:", X_train_risk.shape)
print("Risk Test shape :", X_test_risk.shape)
print("Risk Train period:", risk_train["visit_date"].min().date(), "→", risk_train["visit_date"].max().date())
print("Risk Test period :", risk_test["visit_date"].min().date(), "→", risk_test["visit_date"].max().date())

Risk Train shape: (20000, 14)
Risk Test shape : (5000, 14)
Risk Train period: 2025-01-20 → 2025-11-08
Risk Test period : 2025-11-08 → 2026-01-20


In [9]:
# Step 6) Time Based Split - For Claim

claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)

split_idx = int(len(claim_df) * 0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test  = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_features]
X_test_claim  = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim  = claim_test[claim_target]

print("Claim Train shape:", X_train_claim.shape)
print("Claim Test shape :", X_test_claim.shape)
print("Claim Train period:", claim_train["billing_date"].min().date(), "→", claim_train["billing_date"].max().date())
print("Claim Test period :", claim_test["billing_date"].min().date(), "→", claim_test["billing_date"].max().date())

Claim Train shape: (20000, 18)
Claim Test shape : (5000, 18)
Claim Train period: 2025-01-20 → 2025-11-10
Claim Test period : 2025-11-10 → 2026-01-20


In [10]:
# Step 7) Log Risk Model
with mlflow.start_run(run_name="RandomForest-Risk") as run:
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "class_weight": "balanced_subsample",
        "evaluation_data": "risk_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "visit_date"
    })

    pred_risk = risk_rf_model.predict(X_test_risk)

    # Metrics
    acc_risk = accuracy_score(y_test_risk, pred_risk)
    f1_risk = f1_score(y_test_risk, pred_risk, average="weighted")
    high_recall = recall_score(y_test_risk, pred_risk, labels=["High"], average=None)[0]

    # Logging the Metrics
    mlflow.log_metric("accuracy", acc_risk)
    mlflow.log_metric("weighted_f1", f1_risk)
    mlflow.log_metric("high_risk_recall", high_recall)

    # Logging the Model
    mlflow.sklearn.log_model(
    sk_model=risk_rf_model,
    name="model",
    serialization_format="pickle"
)

    # Fetching the run id
    risk_run_id = run.info.run_id

    print("RandomForest Risk Model logged ✓")
    print(f"  Accuracy         : {acc_risk:.4f}")
    print(f"  Weighted F1      : {f1_risk:.4f}")
    print(f"  High Risk Recall : {high_recall:.4f}")
    print(f"  Run ID           : {risk_run_id}")


2026/09/14 12:36:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/14 12:36:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest Risk Model logged ✓
  Accuracy         : 0.4064
  Weighted F1      : 0.3986
  High Risk Recall : 0.1500
  Run ID           : a0497189af7e4904a31789e24baf09bb


In [11]:
# Step-8) Log Claim Model
with mlflow.start_run(run_name="RandomForest-Claim") as run:

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 250,
        "max_depth": 14,
        "min_samples_split": 8,
        "class_weight": "balanced",
        "evaluation_data": "claim_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "billing_date"
    })

    pred_claim = claim_rf_model.predict(X_test_claim)

    acc_claim = accuracy_score(y_test_claim, pred_claim)
    f1_claim = f1_score(y_test_claim, pred_claim, average="weighted")
    rejected_recall = recall_score(
        y_test_claim, pred_claim, labels=["Rejected"], average=None
    )[0]

    mlflow.log_metric("accuracy", acc_claim)
    mlflow.log_metric("weighted_f1", f1_claim)
    mlflow.log_metric("rejected_recall", rejected_recall)

    mlflow.sklearn.log_model(
    sk_model=risk_rf_model,
    name="model",
    serialization_format="pickle"
)

    claim_run_id = run.info.run_id

    print("RandomForest Claim Model logged ✓")
    print(f"  Accuracy         : {acc_claim:.4f}")
    print(f"  Weighted F1      : {f1_claim:.4f}")
    print(f"  Rejected Recall  : {rejected_recall:.4f}")
    print(f"  Run ID           : {claim_run_id}")

2026/09/14 12:36:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/14 12:36:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


RandomForest Claim Model logged ✓
  Accuracy         : 0.4260
  Weighted F1      : 0.4317
  Rejected Recall  : 0.6545
  Run ID           : 4bf046a111a949b0b1cb05fe6531ba37


In [12]:
# Step-9) Print the final summary
print("=" * 60)
print("MLFLOW RUNS SUMMARY")
print("=" * 60)
print(f"Risk  Model Run ID : {risk_run_id}")
print(f"Claim Model Run ID : {claim_run_id}")
print()
print("Open MLflow UI at: http://127.0.0.1:5000")

MLFLOW RUNS SUMMARY
Risk  Model Run ID : a0497189af7e4904a31789e24baf09bb
Claim Model Run ID : 4bf046a111a949b0b1cb05fe6531ba37

Open MLflow UI at: http://127.0.0.1:5000


### Register the Model

Registration creates a named, versioned entry
in the MLflow Model Registry.

Every time you register the same name,
the version number increments automatically.
v1 → v2 → v3 and so on.

In [13]:
# Step 1) Register the Risk RF Model in Model Registry
from mlflow import register_model

register_model_name = "HealthRiskRFModel"

model_uri = f"runs:/{risk_run_id}/model"

result = register_model(
    model_uri=model_uri,
    name=register_model_name
)

print("Register Model Name: ", result.name)
print("Registered version: ", result.version)

Successfully registered model 'HealthRiskRFModel'.
2026/09/14 12:36:24 WARNING mlflow.tracking._model_registry.fluent: Run with id a0497189af7e4904a31789e24baf09bb has no artifacts at artifact path 'model', registering model based on models:/m-0e54b0a0e93642ec84be608b1d7be04d instead


Register Model Name:  HealthRiskRFModel
Registered version:  1


Created version '1' of model 'HealthRiskRFModel'.


In [14]:
# Step 2) Saving the Registered version in a variable
risk_model_version = result.version
print("Risk Model Version Saved: ", risk_model_version)

Risk Model Version Saved:  1


In [15]:
# Step 3) Create MLflow Client
from mlflow.tracking import MlflowClient
client = MlflowClient()
print("MLflow Client ready ✓")

MLflow Client ready ✓


In [16]:
# Step 4) Move Risk Model version to Staging
client.transition_model_version_stage(
    name=register_model_name,
    version=risk_model_version,
    stage="Staging"
)
print(f"Model {register_model_name} version {risk_model_version} moved to staging")

Model HealthRiskRFModel version 1 moved to staging


In [17]:
#  Step 5) Check if model qualifies for Production
if acc_risk >= 0.55 and high_recall >=0.70:
    print("Risk Model is eligible for Production Promotion ✓")
else:
    print("Risk Model is not eligible")

Risk Model is not eligible


In [18]:
# Step 6) Promote the Risk Model to Production
client.transition_model_version_stage(
    name=register_model_name,
    version=risk_model_version,
    stage="Production",
    archive_existing_versions=True
)

print(f"Model {register_model_name} version {risk_model_version} moved to Production ✓")

Model HealthRiskRFModel version 1 moved to Production ✓


In [19]:
# Step 7) Load the Production Risk Model from Registry
import mlflow.sklearn

production_risk_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{register_model_name}/Production"
)
# models:/HealthcareRiskRFModel/Production
print("Production Risk Model Loaded ✓")

Production Risk Model Loaded ✓


In [20]:
# Step 8) Fetch the current Production version
latest_versions = client.get_latest_versions(
    name=register_model_name,
    stages=["Production"]
)

production_version = latest_versions[0].version if latest_versions else None
print("Current Production Version: ", production_version)

Current Production Version:  1


In [21]:
# Step 9) Run a prediction using the Production Risk Model
pred_risk = production_risk_model.predict(X_test_risk.head(5))
print("Predictions: ", pred_risk)

Predictions:  ['Medium' 'Low' 'High' 'Low' 'Medium']


In [22]:
# Step 10) Log Prediction with Model Version
import hashlib
from datetime import datetime

def hash_input(payload: dict) -> str:
    payload_str = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload_str.encode()).hexdigest()

In [23]:
# Step 11) Input Payload
input_payload = {
    "age": 52,
    "gender": "M",                        
    "city": "Bangalore",
    "insurance_provider": "CareOne",       
    "chronic_flag": 1,
    "department": "Cardiology",
    "visit_type": "ER",                    
    "doctor_id": 101,
    "length_of_stay_hours": 48,
    "days_since_registration": 300,
    "visit_frequency": 4,
    "avg_los_per_patient": 36.5,
    "visit_month": 3,
    "visit_dayofweek": 2
}

In [24]:
# Step 12) Generate Hash
input_hash = hash_input(input_payload)
print("Input Hash: ", input_hash)

Input Hash:  393af61ee47c8f37fa64b93931d86baa4a2690fc9a9ecf27987b660d34e5c8bd


In [25]:
# Step 13) Creating a Log Record 

# Step 1: Define logs directory
BASE_DIR = os.getcwd()  # or project root if running from notebooks
LOG_DIR = os.path.join(BASE_DIR, "logs")

# Step 2: Create logs folder if not exists
os.makedirs(LOG_DIR, exist_ok=True)

# Step 3: Define log file path
LOG_FILE = os.path.join(LOG_DIR, "predictions.log")

# Step 4:
prediction_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_name": register_model_name,
    "model_version": production_version,
    "input_hash": input_hash,
    "prediction": str(pred_risk[0])
}

with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(json.dumps(prediction_log) + "\n")

print("Prediction logged with model_version ✓")

Prediction logged with model_version ✓
